## Q1. Embedding a query

Embed the following query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

* ❌ -0.31
* ✅ -0.02
* ❌ 0.12
* ❌ 0.44

In [1]:
from typing import Protocol, cast

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

from lib.types import EmbeddingVector, LessonChunk, LessonDocument
from scripts.embedder import Embedder


class VectorSearchIndex(Protocol):
    def fit(
        self,
        vectors: EmbeddingVector,
        payload: list[LessonChunk],
    ) -> "VectorSearchIndex": ...

    def search(
        self,
        query_vector: EmbeddingVector,
        num_results: int = 10,
    ) -> list[LessonChunk]: ...


class LexicalSearchIndex(Protocol):
    def fit(self, docs: list[LessonChunk]) -> "LexicalSearchIndex": ...

    def search(
        self,
        query: str,
        num_results: int = 10,
    ) -> list[LessonChunk]: ...

In [2]:
embedder = Embedder()

In [3]:
question_1_query = "How does approximate nearest neighbor search work?"

In [4]:
question_1_query_embedding = embedder.encode(question_1_query)

print(question_1_query_embedding[0])

-0.02058203437252893


### Loading the data

We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = cast(list[LessonDocument], [file.parse() for file in reader.read()])

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page [02-vector-search/lessons/07-sqlitesearch-vector.md](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/07-sqlitesearch-vector.md), embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

* ❌ 0.07
* ✅ 0.37
* ❌ 0.68
* ❌ 0.92

In [6]:
predicate_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

lesson_content = next(
    document["content"]
    for document in documents
    if document["filename"] == predicate_filename
)

In [7]:
lesson_content_embedding = embedder.encode(lesson_content)

In [8]:
lesson_content_embedding.dot(question_1_query_embedding)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

```python
scores = X.dot(v)
```

Which file does the highest-scoring chunk belong to (its filename)?

* ❌ 02-vector-search/lessons/03-embeddings-dataset.md
* ❌ 02-vector-search/lessons/06-rag-vector.md
* ✅ 02-vector-search/lessons/07-sqlitesearch-vector.md
* ❌ 02-vector-search/lessons/09-onnx-embedder.md

In [9]:
chunks = cast(
    list[LessonChunk],
    chunk_documents(
        cast(list[dict[str, str]], documents),
        size=2000,
        step=1000,
    ),
)

In [10]:
texts: list[str] = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(texts)

In [11]:
scores = X.dot(question_1_query_embedding)

In [12]:
chunks[scores.argmax()]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use `VectorSearch` from minsearch and run a search for the following query:

> What metric do we use to evaluate a search engine?

Which file is the `filename` of the first result?

* ❌ 02-vector-search/lessons/04-vector-search.md
* ✅ 04-evaluation/lessons/05-search-metrics.md
* ❌ 04-evaluation/lessons/13-llm-as-judge.md
* ❌ 05-monitoring/lessons/04-metrics.md

In [13]:
question_4_query = "What metric do we use to evaluate a search engine?"

semantic_index = cast(
    VectorSearchIndex,
    VectorSearch(keyword_fields=["filename"]),
)

semantic_index.fit(X, chunks)

In [14]:
question_4_query_embedding = embedder.encode(question_4_query)

question_4_semantic_results = semantic_index.search(
    question_4_query_embedding,
    num_results=5,
)

In [15]:
question_4_semantic_results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use `content` as a text field.

Run both searches for this query:

> How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

* ❌ 02-vector-search/lessons/01-intro.md
* ❌ 02-vector-search/lessons/02-embeddings.md
* ✅ 02-vector-search/lessons/08-pgvector.md
* ❌ 03-orchestration/lessons/05-rag.md

In [16]:
lexical_index = cast(
    LexicalSearchIndex,
    Index(text_fields=["content"]),
)

lexical_index.fit(chunks)

In [17]:
question_5_query = "How do I store vectors in PostgreSQL?"

In [18]:
question_5_lexical_results = lexical_index.search(question_5_query, num_results=5)

In [19]:
question_5_query_embedding = embedder.encode(question_5_query)

question_5_semantic_results = semantic_index.search(question_5_query_embedding, num_results=5)

In [20]:
question_5_only_semantic_filenames = (
    {result["filename"] for result in question_5_semantic_results}
    - {result["filename"] for result in question_5_lexical_results}
)

print(question_5_only_semantic_filenames)

{'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid search

Keyword search is great for exact terms, and vector search is great for meaning. In practice, hybrid search often works better than either one alone.

We can combine the top results from both retrieval methods with Reciprocal Rank Fusion (RRF).

Run hybrid search for this query:

> How do I give the model access to tools?

Which file is the `filename` of the first result?

* ✅ 01-agentic-rag/lessons/13-function-calling.md
* ❌ 01-agentic-rag/lessons/14-agentic-loop.md
* ❌ 01-agentic-rag/lessons/15-frameworks.md
* ❌ 03-orchestration/lessons/06-agents.md

In [21]:
def rrf(
    result_lists: list[list[LessonChunk]],
    k: int = 60,
    num_results: int = 5,
) -> list[LessonChunk]:
    scores: dict[tuple[str, int], float] = {}
    docs: dict[tuple[str, int], LessonChunk] = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0.0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=lambda key: scores[key], reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [22]:
question_6_query = "How do I give the model access to tools?"

In [23]:
question_6_lexical_results = (
    lexical_index.search(question_6_query, num_results=5))

In [24]:
question_6_query_embedding = embedder.encode(question_6_query)

In [25]:
question_6_semantic_results = (
    semantic_index.search(question_6_query_embedding, num_results=5))

In [26]:
question_6_results = rrf([
    question_6_lexical_results,
    question_6_semantic_results])

In [27]:
question_6_results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'